In [26]:
import sys
import os
project_root = os.path.abspath("..")
sys.path.append(project_root)

import torch
import yaml
import re
import glob
from pathlib import Path

from main import LVSMLauncherConfig
from src.models.lvsm_decoder_only import LVSMDecoderOnlyModel
from src.configs.lvsm_decoder_only_config import (
    LVSMDecoderOnlyModelConfig,
    RayEncodingType,
    PosEncType,
)
from src.prope.utils.transformer import (
    TransformerEncoderConfig,
    TransformerEncoderLayerConfig,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

run_dir = "../results/nvs/PX/release-1gpus-b8-s1-80k-CAMRAY-PROPE/" #PX
# run_dir = "../results/nvs/release-1gpus-b8-s1-80k-CAMRAY-PROPE/" #VAE



def get_config(dir):
    # -------------------------------
    # Load & clean YAML
    # -------------------------------    
    raw = Path(f"{dir}/config.yaml").read_text()
    clean = re.sub(r"!!python/[^ \n]+", "", raw)   # strip python tags

    config_dict = yaml.safe_load(clean)


    # -------------------------------
    # Rebuild the launcher config
    # -------------------------------
    cfg = LVSMLauncherConfig()

    for k, v in config_dict.items():
        if k != "model_config":   # we'll rebuild this separately
            setattr(cfg, k, v)


    # -------------------------------
    # Rebuild the nested model config
    # -------------------------------
    m = config_dict["model_config"]

    # Enums were dumped as single-item lists
    ray_encoding = RayEncodingType(m["ray_encoding"][0])
    pos_enc      = PosEncType(m["pos_enc"][0])

    # Encoder layer
    layer_cfg = m["encoder"]["layer"]

    encoder_layer = TransformerEncoderLayerConfig(
        d_model=layer_cfg["d_model"],
        nhead=layer_cfg["nhead"],
        dim_feedforward=layer_cfg["dim_feedforward"],
        dropout=layer_cfg["dropout"],
        activation=torch.nn.functional.relu,  # original activation
        batch_first=layer_cfg["batch_first"],
        bias=layer_cfg["bias"],
        layer_norm_eps=layer_cfg["layer_norm_eps"],
        modulation_activation=layer_cfg["modulation_activation"],
        norm_first=layer_cfg["norm_first"],
        norm_type=layer_cfg["norm_type"],
        elementwise_affine=layer_cfg["elementwise_affine"],
        qk_norm=layer_cfg["qk_norm"],
    )

    encoder = TransformerEncoderConfig(
        layer=encoder_layer,
        num_layers=m["encoder"]["num_layers"],
        input_norm=m["encoder"]["input_norm"],
        output_norm=m["encoder"]["output_norm"],
        checkpointing=m["encoder"]["checkpointing"],
    )

    # Full model config
    model_cfg = LVSMDecoderOnlyModelConfig(
        ref_views=m["ref_views"],
        tar_views=m["tar_views"],
        encoder=encoder,
        img_shape=tuple(m["img_shape"]),
        cam_shape=tuple(m["cam_shape"]),
        patch_size=m["patch_size"],
        ray_encoding=ray_encoding,
        pos_enc=pos_enc,
    )

    # The launcher should store this model_config
    cfg.model_config = model_cfg
    return cfg

def load_checkpoint(model, dir):
    def get_latest_checkpoint(ckpt_dir):
        ckpts = sorted(glob.glob(os.path.join(ckpt_dir, "*.pt")))
        return ckpts[-1] if ckpts else None

    ckpt_path = get_latest_checkpoint(f"{dir}/ckpts")
    print("Loading checkpoint:", ckpt_path)

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    state_dict = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model"].items()}
    missing, unexpected = model.load_state_dict(state_dict, strict=False)

    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)
    return model

cfg = get_config(run_dir)

model = LVSMDecoderOnlyModel(cfg.model_config).to(device)
# model = load_checkpoint(model, run_dir)
model.eval()

print("Model space:", cfg.model_space)

Model space: PX


In [27]:
import glob

from src.data.dataset import TrainDataset
from src.data.latent_dataset import TrainLatentDataset
from main import LVSMLauncher, LVSMLauncherConfig


def load_single_batch(cfg: LVSMLauncherConfig):
    """
    Loads ONE batch from the appropriate dataset based on cfg.model_space.
    Returns ref_imgs, tar_imgs, ref_cams, tar_cams, ref_paths, tar_paths.
    """

    launcher = LVSMLauncher(cfg)
    
    if cfg.model_space == "PX":
        scenes = sorted(glob.glob("../data/data_processed/realestate10k/train/*"))
        dataset = TrainDataset(
            scenes,
            patch_size=cfg.dataset_patch_size,
            zoom_factor=cfg.train_zoom_factor,
            random_zoom=cfg.random_zoom,
            supervise_views=cfg.dataset_supervise_views,
        )
    else: 
        scenes = sorted(glob.glob("../data/data_processed/realestate10k_latent/train/*"))
        dataset = TrainLatentDataset(
            scenes,
            supervise_views=cfg.dataset_supervise_views,
        )

    data = dataset[0] # just one item
    data["image"] = data["image"].unsqueeze(0)
    data["K"] = data["K"].unsqueeze(0)
    data["camtoworld"] = data["camtoworld"].unsqueeze(0)
    data["image_path"] = [data["image_path"]]  # wrap in list
    
    input_views = data["K"].shape[1] - cfg.dataset_supervise_views
    
    processed = launcher.preprocess(data, input_views=input_views)

    ref_imgs = processed["ref_imgs"]
    tar_imgs = processed["tar_imgs"]
    ref_cams = processed["ref_cams"]
    tar_cams = processed["tar_cams"]
    ref_paths = processed["ref_paths"]
    tar_paths = processed["tar_paths"]

    return ref_imgs, tar_imgs, ref_cams, tar_cams, ref_paths, tar_paths


In [28]:
from main import LVSMLauncherConfig



ref_imgs, tar_imgs, ref_cams, tar_cams, ref_paths, tar_paths = load_single_batch(cfg)

print("ref_imgs:", ref_imgs.shape)
print("tar_imgs:", tar_imgs.shape)
print("latent?" if cfg.model_space == "VAE" else "pixels")

Wrote config to results/nvs/PX/release-1gpus-b8-s1-80k-CAMRAY-PROPE/config.yaml
ref_imgs: torch.Size([1, 2, 256, 256, 3])
tar_imgs: torch.Size([1, 1, 256, 256, 3])
pixels


In [29]:
launcher = LVSMLauncher(cfg)

with torch.inference_mode():
    out = model(ref_imgs.to(device), ref_cams, tar_cams)

if cfg.model_space == "PX":
    out = torch.sigmoid(out)
else:
    img = launcher.decode_tensors(out[0])

out

Wrote config to results/nvs/PX/release-1gpus-b8-s1-80k-CAMRAY-PROPE/config.yaml


tensor([[[[[0.6003, 0.6149, 0.3409],
           [0.6001, 0.4771, 0.6270],
           [0.7235, 0.4738, 0.4884],
           ...,
           [0.5098, 0.4177, 0.3544],
           [0.6115, 0.3709, 0.3764],
           [0.6929, 0.5939, 0.5968]],

          [[0.4752, 0.4181, 0.2173],
           [0.5197, 0.3325, 0.4814],
           [0.4969, 0.3881, 0.4694],
           ...,
           [0.4513, 0.6588, 0.6310],
           [0.3361, 0.6606, 0.2566],
           [0.3770, 0.4871, 0.5970]],

          [[0.5986, 0.4702, 0.4145],
           [0.5066, 0.4991, 0.5113],
           [0.4723, 0.4342, 0.3106],
           ...,
           [0.4354, 0.4867, 0.8322],
           [0.3976, 0.7065, 0.7048],
           [0.5009, 0.5879, 0.6291]],

          ...,

          [[0.5396, 0.4054, 0.5374],
           [0.4679, 0.5388, 0.3979],
           [0.4807, 0.6262, 0.4938],
           ...,
           [0.2965, 0.3269, 0.4065],
           [0.4704, 0.5709, 0.6151],
           [0.4411, 0.2550, 0.2083]],

          [[0.5319, 0.48

In [30]:
import time
import torch

def tensor_bytes(x: torch.Tensor) -> int:
    return x.numel() * x.element_size()


def run_inference_with_metrics(model, launcher, ref_imgs, tar_cams, cfg, device="cuda"):

    # ------------------------------------------
    # Model parameter stats
    # ------------------------------------------
    model_params = sum(p.numel() for p in model.parameters())

    # ------------------------------------------
    # Input size (bytes)
    # ------------------------------------------
    input_tensors = [ref_imgs]
    input_bytes = sum(tensor_bytes(t) for t in input_tensors if isinstance(t, torch.Tensor))

    # These final numbers we will fill:
    forward_memory_mb = 0.0
    decode_memory_mb = 0.0
    peak_memory_mb = 0.0

    # ------------------------------------------
    # Forward pass timing + memory
    # ------------------------------------------
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)

    torch.cuda.synchronize()
    t_fwd0 = time.time()

    with torch.inference_mode():
        out = model(ref_imgs.to(device), ref_cams, tar_cams)

    torch.cuda.synchronize()
    t_fwd1 = time.time()

    forward_time_ms = (t_fwd1 - t_fwd0) * 1000

    # Memory AFTER forward pass
    if torch.cuda.is_available():
        forward_memory_mb = torch.cuda.max_memory_allocated(device) / (1024**2)

    # Keep the forward peak BEFORE decoding wipes it out
    forward_peak = torch.cuda.max_memory_allocated(device)

    # ------------------------------------------
    # Decoding (only for VAE)
    # ------------------------------------------
    if cfg.model_space == "VAE":

        # Reset for decode-only measurement
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize()
        t_dec0 = time.time()

        decoded = launcher.decode_tensors(out[0])

        torch.cuda.synchronize()
        t_dec1 = time.time()
        decode_time_ms = (t_dec1 - t_dec0) * 1000

        decode_memory_mb = torch.cuda.max_memory_allocated(device) / (1024**2)

    else:
        decoded = torch.sigmoid(out)
        decode_time_ms = 0.0
        decode_memory_mb = 0.0

    # ------------------------------------------
    # Total peak memory (full process)
    # ------------------------------------------
    peak_memory_mb = max(forward_peak, torch.cuda.max_memory_allocated(device)) / (1024**2)

    # ------------------------------------------
    # Total wall time
    # ------------------------------------------
    total_time_ms = forward_time_ms + decode_time_ms

    # ------------------------------------------
    # Return metrics
    # ------------------------------------------
    return {
        "model_params": model_params,
        "input_bytes": input_bytes,
        "input_MB": input_bytes / (1024**2),
        "forward_time_ms": forward_time_ms,
        "decode_time_ms": decode_time_ms,
        "total_time_ms": total_time_ms,
        "forward_memory_mb": forward_memory_mb,
        "decode_memory_mb": decode_memory_mb,
        "peak_memory_mb": peak_memory_mb,
        "output_shape": tuple(out.shape),
        "decoded_shape": tuple(decoded.shape),
    }


In [31]:
launcher = LVSMLauncher(cfg)

metrics = run_inference_with_metrics(
    model=model,
    launcher=launcher,
    ref_imgs=ref_imgs,
    tar_cams=tar_cams,
    cfg=cfg,
)

print(metrics)


Wrote config to results/nvs/PX/release-1gpus-b8-s1-80k-CAMRAY-PROPE/config.yaml
{'model_params': 16621440, 'input_bytes': 1572864, 'input_MB': 1.5, 'forward_time_ms': 103.04474830627441, 'decode_time_ms': 0.0, 'total_time_ms': 103.04474830627441, 'forward_memory_mb': 589.4140625, 'decode_memory_mb': 0.0, 'peak_memory_mb': 589.4140625, 'output_shape': (1, 1, 256, 256, 3), 'decoded_shape': (1, 1, 256, 256, 3)}
